# Stage 7 — TabPFN, and a cautionary tale about CV

**Question:** does TabPFN (v3.5), the last untested model family, help?

- TabPFN run on the same folds; blended with every saved stage 4–6 prediction.
- An automatic SUBMIT / DON'T SUBMIT verdict based on held-out logloss.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| prob_blend (TabPFN weight 0.84) | **0.27622** (best CV ever) | 0.109558 | 1105 (worst since stage 2) |

**Finding:** the best CV of the project produced one of the worst leaderboard scores. Of the 49 rows that changed vs. stage 5, stage 5 was right on 30. A plausible explanation is pretraining contamination: if a foundation model was trained on real public tables that include this dataset, its out-of-fold predictions on the (public) training rows are optimistic, while the test labels were never public. Unproven, but it is a reason to validate foundation models on data they cannot have seen.

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [1]:
import os, sys, glob, time, inspect, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    # also accepts the file saved as .txt (browsers sometimes do that when downloading CSVs)
    for n in [name, name.replace('.csv', '.txt')]:
        hits = glob.glob(f'/kaggle/input/**/{n}', recursive=True) + glob.glob(n)
        if hits:
            return os.path.abspath(hits[0])
    return None

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'            # quick smoke test
N_FOLDS = 5
TABPFN_SEEDS = [0] if FAST else [0, 1, 2]       # fold splits (same splits as v4-v6)
TABPFN_ESTIMATORS = 2 if FAST else 4
BUDGET_MIN = 45                                 # stops adding seeds past this
CONTEXT_FALLBACK = 10000                        # if the GPU runs out of memory, retry with this many training rows per fit
FULL_REFIT, FULL_WEIGHT = True, 0.5
V5_BEST, V6_BEST = 0.27691, 0.27695             # held-out logloss of the v5 (LB 1094) and v6 blends
SUBMIT_IF_GAIN = 0.0001                         # verdict: submit only if the blend improves by at least this
# ---------------------------------------------------------------------------

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
    print('pip install', *pkgs, '->', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
    return r.returncode == 0

pip_install('tabpfn')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| device:', DEVICE, '| FAST:', FAST)

pip install tabpfn -> ok
train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | device: cuda | FAST: False


## Data (one-hot + numeric, as TabPFN used in v5/v6)

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)
both = pd.concat([X, X_test], ignore_index=True)
OH = pd.get_dummies(both[CATS], dtype=np.float32)
OH[NUMS] = both[NUMS].astype(np.float32)
Xoh, Xoh_test = OH.iloc[:len(X)].reset_index(drop=True), OH.iloc[len(X):].reset_index(drop=True)
print(X.shape, '| one-hot', Xoh.shape[1], '| positive rate %.4f' % y.mean())

(23524, 10) | one-hot 39 | positive rate 0.1408


## Harness (same folds, time budget)

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)) for s in range(10)}
ALL_ROWS = np.arange(len(y))
OOF, TEST_FOLD, TEST_FULL, SCORES, NEW_MODELS = {}, {}, {}, {}, []

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit, seeds, budget_min=None):
    t0, done = time.time(), []
    oof = np.zeros((len(seeds), len(y)))
    tst, full = np.zeros(len(X_test)), np.zeros(len(X_test))
    for k, s in enumerate(seeds):
        if done and budget_min and (time.time() - t0) / 60 * (len(done) + 1) / len(done) > budget_min:
            print(f'  {name}: time budget reached, stopping after {len(done)} seed(s)')
            break
        for tr_idx, va_idx in FOLDS[s]:
            predict = fit(tr_idx, s)
            oof[k, va_idx] = predict(va_idx)
            tst += predict(None) / N_FOLDS
        if FULL_REFIT:
            full += fit(ALL_ROWS, s)(None)
        done.append(k)
        print(f'  {name}: seed {s} done ({(time.time() - t0) / 60:.1f} min)')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    n = len(done)
    oof, tst, full = oof[done], tst / n, (full / n if FULL_REFIT else tst / n)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof.mean(0), tst, full
    SCORES[name] = dict(cv_error=np.mean(per_seed), logloss=np.mean([log_loss(y, o) for o in oof]),
                        logloss_avg=log_loss(y, OOF[name]), seeds=n, minutes=(time.time() - t0) / 60)
    NEW_MODELS.append(name)
    print(f"{name:14s} logloss {SCORES[name]['logloss']:.5f} (seed-avg {SCORES[name]['logloss_avg']:.5f}) | "
          f"CV error {np.mean(per_seed):.5f} | {n} seed(s) | {SCORES[name]['minutes']:.1f} min")

def make(cls, **kw):
    # Pass only the arguments this library version accepts, so API differences don't crash the run.
    params = inspect.signature(cls.__init__).parameters
    if not any(p.kind == p.VAR_KEYWORD for p in params.values()):
        kw = {k: v for k, v in kw.items() if k in params}
    return cls(**kw)

def add_saved(name, oof, fold, full):
    OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof, fold, full
    SCORES[name] = dict(cv_error=error_rate(oof), logloss=np.nan, logloss_avg=log_loss(y, oof), seeds=np.nan, minutes=0.0)

## Load v4, v5 and v6 predictions
Later versions overwrite earlier ones with the same name (v6's `tabm` is the 8-seed average that replaces v5's 3-seed one).

In [4]:
DROP_OLD = ['realmlp']
for ver in ['v4', 'v5', 'v6']:
    paths = [find_file(f) for f in [f'oof_{ver}.csv', f'test_probabilities_{ver}_fold.csv', f'test_probabilities_{ver}_full.csv']]
    if not all(paths):
        print(f'!! {ver} files not found in the inputs. Continuing without them.')
        continue
    o, td, tf = (pd.read_csv(p) for p in paths)
    assert (o['y'].values == y).all(), f'oof_{ver} rows do not match Train.csv'
    assert (td['unique_id'].values == test_ids.values).all() and (tf['unique_id'].values == test_ids.values).all(), \
        f'{ver} test rows do not match Test.csv'
    kept = [c for c in o.columns.drop('y') if c not in DROP_OLD]
    for c in kept:
        add_saved(c, o[c].values, td[c].values, tf[c].values)
    print(f'loaded {ver}:', kept)
if not OOF:
    raise FileNotFoundError('No saved predictions found - attach the fi-predictions dataset.')
if find_file('oof_v6.csv') is None:
    print('NOTE: without v6 the reference to beat is still v5 (0.27691), but the blend has fewer models to work with.')
BASE_MODELS = list(OOF)

loaded v4: ['cb_cpu', 'cb_cpu_v2feat', 'cb_cpu_slow', 'cb_cpu_deep', 'lightgbm', 'lightgbm_xt', 'logreg_cross', 'embed_nn']
loaded v5: ['tabicl', 'tabm', 'cb_optuna']
loaded v6: ['tabicl32', 'tabicl_ord', 'tabm']


## TabPFN
Needs the `TABPFN_TOKEN` secret attached to this notebook **and** the license accepted on ux.priorlabs.ai.
If the GPU runs out of memory on ~19k training rows, it automatically retries with a 10k-row random subset per fit.

In [5]:
token = os.environ.get('TABPFN_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception as e:
        print('Could not read the TABPFN_TOKEN secret:', type(e).__name__, '- tick it under Add-ons -> Secrets.')
        token = None

TABPFN_STATUS = 'not run'
if token:
    os.environ['TABPFN_TOKEN'] = token
    from tabpfn import TabPFNClassifier

    def tabpfn_fit_factory(context_max):
        def fit(tr_idx, seed):
            rows = tr_idx
            if context_max and len(rows) > context_max:
                rows = np.sort(np.random.default_rng(seed).choice(rows, context_max, replace=False))
            m = make(TabPFNClassifier, n_estimators=TABPFN_ESTIMATORS, random_state=seed, device=DEVICE,
                     ignore_pretraining_limits=True)
            m.fit(Xoh.iloc[rows].values, y[rows])
            def predict(r):
                A = Xoh_test.values if r is None else Xoh.iloc[r].values
                return np.concatenate([m.predict_proba(A[i:i + 5000])[:, 1] for i in range(0, len(A), 5000)])
            return predict
        return fit

    for context_max in [None, CONTEXT_FALLBACK]:
        try:
            run_model('tabpfn', tabpfn_fit_factory(context_max), TABPFN_SEEDS, BUDGET_MIN)
            TABPFN_STATUS = 'ok' + (f' (context {context_max} rows)' if context_max else '')
            break
        except Exception as e:
            if 'out of memory' in str(e).lower():
                print(f'!! out of GPU memory with context={context_max or "all rows"}' +
                      (' - retrying with a smaller context' if context_max is None else ''))
                torch.cuda.empty_cache()
                TABPFN_STATUS = 'out of memory'
                continue
            TABPFN_STATUS = f'{type(e).__name__}: {str(e)[:800]}'
            print('!! tabpfn failed:', TABPFN_STATUS)
            if 'icense' in str(e):
                print('-> Accept the license at ux.priorlabs.ai (Licenses tab), then rerun.')
            break
else:
    TABPFN_STATUS = 'no token'
print('TabPFN status:', TABPFN_STATUS)

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

  tabpfn: seed 0 done (9.5 min)
  tabpfn: seed 1 done (19.0 min)
  tabpfn: seed 2 done (28.4 min)
tabpfn         logloss 0.27643 (seed-avg 0.27607) | CV error 0.11153 | 3 seed(s) | 28.4 min
TabPFN status: ok


## Save TabPFN's predictions (for reuse)

In [6]:
if NEW_MODELS:
    pd.DataFrame({'y': y, **{k: OOF[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/oof_v7.csv', index=False)
    pd.DataFrame({'unique_id': test_ids, **{k: TEST_FOLD[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v7_fold.csv', index=False)
    pd.DataFrame({'unique_id': test_ids, **{k: TEST_FULL[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v7_full.csv', index=False)
    ref = 'cb_cpu_deep' if 'cb_cpu_deep' in OOF else BASE_MODELS[0]
    print(f"tabpfn correlation with {ref}: {np.corrcoef(OOF['tabpfn'], OOF[ref])[0, 1]:.4f}")

tabpfn correlation with cb_cpu_deep: 0.9933


## Blend (same combiners as v4–v6) and the verdict

In [7]:
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

class ProbBlend:                       # weighted average of probabilities, weights >= 0 summing to 1
    def fit(self, F, t):
        F = np.clip(F, 1e-6, 1 - 1e-6)
        def loss(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))
        def grad(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -F.T @ (t / p - (1 - t) / (1 - p)) / len(t)
        k = F.shape[1]
        r = minimize(loss, np.ones(k) / k, jac=grad, method='SLSQP', bounds=[(0, 1)] * k,
                     constraints=[dict(type='eq', fun=lambda w: w.sum() - 1, jac=lambda w: np.ones(k))],
                     options=dict(maxiter=500, ftol=1e-10))
        self.w = np.clip(r.x, 0, None) / np.clip(r.x, 0, None).sum()
        return self
    def predict(self, F):
        return F @ self.w

class LogitBlend:
    def fit(self, F, t):
        self.m = LogisticRegression(C=1.0, max_iter=5000).fit(logit(F), t); return self
    def predict(self, F):
        return self.m.predict_proba(logit(F))[:, 1]

CANDIDATES = {'prob_blend': ProbBlend, 'logit_blend': LogitBlend}
META_SEEDS = [2024, 7, 99]

def held_out(make, F):
    out = np.zeros(len(y))
    for ms in META_SEEDS:
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=ms).split(F, y):
            out[va] += make().fit(F[tr], y[tr]).predict(F[va]) / len(META_SEEDS)
    return out

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
HELD_OOF = {k: held_out(make, P) for k, make in CANDIDATES.items()}
without = log_loss(y, held_out(ProbBlend, np.column_stack([OOF[n] for n in BASE_MODELS])))
for k in CANDIDATES:
    print(f'{k:12s} held-out logloss {log_loss(y, HELD_OOF[k]):.5f} | error {error_rate(HELD_OOF[k]):.5f}')
print(f'same blend without TabPFN: {without:.5f}')

T_MIX = np.column_stack([(1 - FULL_WEIGHT) * TEST_FOLD[n] + FULL_WEIGHT * TEST_FULL[n] for n in names])
FITTED = {k: make().fit(P, y) for k, make in CANDIDATES.items()}
best = min(CANDIDATES, key=lambda k: log_loss(y, HELD_OOF[k]))
best_ll = log_loss(y, HELD_OOF[best])
print('prob_blend weights:', {n: round(float(w), 3) for n, w in zip(names, FITTED['prob_blend'].w) if w > 0.0005})

gain = min(V5_BEST, V6_BEST, without) - best_ll   # must beat v5, v6 and the same blend without TabPFN
print('\n' + '=' * 70)
print(f'TabPFN: {TABPFN_STATUS}')
print(f'best blend ({best}) held-out logloss {best_ll:.5f}  vs  v5 {V5_BEST:.5f}  ->  gain {gain:+.5f}')
if 'tabpfn' in NEW_MODELS and gain >= SUBMIT_IF_GAIN:
    VERDICT = 'SUBMIT submission_v7_best.csv'
else:
    VERDICT = "DON'T SUBMIT - keep v5_best (no meaningful improvement)"
print('VERDICT:', VERDICT)
print('=' * 70)

prob_blend   held-out logloss 0.27622 | error 0.11163
logit_blend  held-out logloss 0.27650 | error 0.11159
same blend without TabPFN: 0.27695
prob_blend weights: {'cb_cpu_deep': 0.014, 'lightgbm': 0.066, 'logreg_cross': 0.001, 'cb_optuna': 0.064, 'tabicl32': 0.015, 'tabpfn': 0.84}

TabPFN: ok
best blend (prob_blend) held-out logloss 0.27622  vs  v5 0.27691  ->  gain +0.00069
VERDICT: SUBMIT submission_v7_best.csv


## Write the submission

In [8]:
sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (FITTED[best].predict(T_MIX) > 0.5).astype(int)})
sub.to_csv(f'{OUT_DIR}/submission_v7_best.csv', index=False)
print(f'submission_v7_best.csv: {sub.bank_account.sum()} ones / {len(sub)} rows   ({VERDICT})')

prev_path = find_file('submission_v5_best.csv')
if prev_path:
    prev = pd.read_csv(prev_path).set_index('unique_id')['bank_account'].reindex(test_ids).values
    print(f'differs from v5_best on {(sub.bank_account.values != prev).sum()} rows')

submission_v7_best.csv: 723 ones / 10086 rows   (SUBMIT submission_v7_best.csv)
